In [ ]:
import os
import random
import torch
import copy
import numpy as np
from torchvision import models
from torchvision.models import (
    ResNet34_Weights,
    ViT_B_16_Weights
)
import timm
import torch.nn as nn
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import torchvision
from torch.utils.data import Subset, DataLoader
import torchvision.transforms as T
from tqdm import tqdm
import json
import matplotlib.pyplot as plt

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Available device: {device}")

In [ ]:
# ============================================================================
# PATHS & DIRECTORIES
# ============================================================================
BASE_DIR = os.path.dirname(os.getcwd())
DATA = os.path.join(BASE_DIR, "data")

# ============================================================================
# STUDENT CE-ONLY PATH (set this manually)
# ============================================================================
STUDENT_CE_ONLY_PATH = os.path.join(
    BASE_DIR,
    "model",
    "ResNet18",
    "CIFAR10",
    "seed_42",
    "split_0.2",
    "val_acc=94.25%_val_loss=0.1993_lr=0.05_wd=0.0005_ep=100_bs=32_opt=SGD_sch=CosineAnnealingLR"
)

# ============================================================================
# EXTRACT PARAMETERS FROM STUDENT CE-ONLY metrics.json
# ============================================================================
_student_metrics_path = os.path.join(STUDENT_CE_ONLY_PATH, "metrics.json")

with open(_student_metrics_path, "r") as f:
    _student_metrics = json.load(f)

print(f"[OK] Student CE-only metrics loaded from: {_student_metrics_path}\n")

# --- Model & Dataset ---
STUDENT_NAME = _student_metrics["model_name"]
DATASET_NAME = _student_metrics["dataset_name"]
SEED = _student_metrics["seed"]
SPLIT = _student_metrics["split"]

# --- Student CE-only Test Accuracy ---
TEST_STUDENT_CE_ACC = _student_metrics["test_acc"]

if DATASET_NAME == "CIFAR10":
    NUM_CLASSES = 10
elif DATASET_NAME == "CIFAR100":
    NUM_CLASSES = 100
elif DATASET_NAME == "Food101":
    NUM_CLASSES = 101
elif DATASET_NAME == "Flowers102":
    NUM_CLASSES = 102
else:
    raise ValueError(f"Unsupported dataset: {DATASET_NAME}")
    

IMG_SIZE = 224

# --- Training Hyperparameters ---
EPOCHS = _student_metrics["epochs"]
BATCH_SIZE = _student_metrics["batch_size"]

# --- Optimizer ---
OPTIMIZER_TYPE = _student_metrics["optimizer"]["type"]
LR = _student_metrics["optimizer"]["params"]["lr"]
WD = _student_metrics["optimizer"]["params"]["weight_decay"]
MOMENTUM = _student_metrics["optimizer"]["params"].get("momentum", 0.9)

# --- Scheduler ---
SCHEDULER_TYPE = _student_metrics["scheduler"]["type"]
_scheduler_params = _student_metrics["scheduler"]["params"]

# --- Augmentations / Transforms ---
TRAIN_TRANSFORMS_CFG = _student_metrics["transforms_train"]
TEST_TRANSFORMS_CFG = _student_metrics["transforms_test"]

# --- Loss ---
LOSS_TYPE = _student_metrics.get("loss_function", {}).get("type", "CrossEntropyLoss")
LABEL_SMOOTHING = _student_metrics.get("loss_function", {}).get("params", {}).get("label_smoothing", 0.0)

# --- Set seed with extracted value ---
set_seed(SEED)

# --- Print extracted parameters ---
print(f"  STUDENT_NAME          = {STUDENT_NAME}")
print(f"  DATASET_NAME          = {DATASET_NAME}")
print(f"  SEED                  = {SEED}")
print(f"  SPLIT                 = {SPLIT}")
print(f"  TEST_STUDENT_CE_ACC   = {TEST_STUDENT_CE_ACC}")
print(f"  EPOCHS                = {EPOCHS}")
print(f"  BATCH_SIZE            = {BATCH_SIZE}")
print(f"  OPTIMIZER_TYPE        = {OPTIMIZER_TYPE}")
print(f"  LR                    = {LR}")
print(f"  WD                    = {WD}")
print(f"  MOMENTUM              = {MOMENTUM}")
print(f"  SCHEDULER_TYPE        = {SCHEDULER_TYPE}")
print(f"  LOSS_TYPE             = {LOSS_TYPE}")
print(f"  Train transforms: {TRAIN_TRANSFORMS_CFG}")
print(f"  Test transforms:  {TEST_TRANSFORMS_CFG}")

# ============================================================================
# TEACHER PATH (set this manually)
# ============================================================================
TEACHER_NAME = "ResNet34"

TEACHER_PATH = os.path.join(
    BASE_DIR,
    "model",
    TEACHER_NAME,
    DATASET_NAME,
    f"seed_{SEED}",
    f"split_{SPLIT}",
    "val_acc=96.70%_val_loss=0.1164_lr=0.01_wd=0.0005_ep=100_bs=128_opt=SGD_sch=CosineAnnealingLR"
)

# ============================================================================
# PRETRAINED MODEL FLAG
# ============================================================================
PRETRAINED_MODEL = False

# ============================================================================
# SELECTION METHOD 
# ============================================================================
METHOD = "FewMedoids_feature_vectors"

MULTI_SEED_METHODS = {"KCenter_feature_vectors"}

# ============================================================================
# SELECTION & SPLIT PATHS
# ============================================================================
SPLIT_DIR = os.path.join(BASE_DIR, "split", DATASET_NAME, f"seed_{SEED}", f"split_{SPLIT}")


DETERMINISTIC_SELECTION_PATH = os.path.join(BASE_DIR, "selection", DATASET_NAME, f"methods_{TEACHER_NAME}", METHOD)

RANDOM_SELECTION_PATH = os.path.join(BASE_DIR, "selection", DATASET_NAME, "Random")

MULTI_SEED_SELECTION_PATH = os.path.join(BASE_DIR, "selection", DATASET_NAME, f"methods_{TEACHER_NAME}", METHOD)

# ============================================================================
# DATA SELECTION & EXPERIMENT CONFIGURATION
# ============================================================================
K_LIST = [1, 2, 4, 8, 16, 32, 64, 128]
SEEDS = [0, 1, 2, 3, 4]
TSNE_MAX_SAMPLES = 2000

# ============================================================================
# KNOWLEDGE DISTILLATION CONFIGURATION
# ============================================================================
TEMP = 4.0
SOFT_TARGET_LOSS_WEIGHT = 0.6
CRITERION_LOSS_WEIGHT = 0.4
FEATURE_MAP_WEIGHT = 0.4
DISTILLATION_TYPE = "soft_label"

# ============================================================================
# LEARNING RATE SCHEDULER PARAMETERS (extracted from metrics)
# ============================================================================
# StepLR
STEP_SIZE = _scheduler_params.get("step_size", 30)
GAMMA = _scheduler_params.get("gamma", 0.1)

# MultiStepLR
MILESTONES = _scheduler_params.get("milestones", [10, 15])

# CosineAnnealingLR
T_MAX = _scheduler_params.get("t_max", EPOCHS)
ETA_MIN = _scheduler_params.get("eta_min", 0.0)

# CosineAnnealingWarmRestarts
T_0 = _scheduler_params.get("t_0", 10)
T_MULT = _scheduler_params.get("t_mult", 2)

# ReduceLROnPlateau
FACTOR = _scheduler_params.get("factor", 0.1)
PATIENCE = _scheduler_params.get("patience", 5)
THRESHOLD = _scheduler_params.get("threshold", 1e-4)
MIN_LR = _scheduler_params.get("min_lr", 1e-6)

print(f"\n[OK] K_LIST = {K_LIST}")
print(f"[OK] ROOT_DIR & EXPERIMENT_DIR will be created per K after training")

In [ ]:
if DISTILLATION_TYPE not in ["soft_label","feature_map"]:
    
    raise ValueError(f"Unsupported distillation type: {DISTILLATION_TYPE}")

In [ ]:
if METHOD not in ["FewMedoids_feature_vectors","KCenter_feature_vectors","Herding_feature_vectors","PCA_Guided_Matching","Random"]:
    
    raise ValueError(f"Unsupported selection method: {METHOD}")



In [ ]:
def create_loss():

    if LOSS_TYPE == "CrossEntropyLoss":
        return nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING
        )

    if LOSS_TYPE == "BCEWithLogitsLoss":
        return nn.BCEWithLogitsLoss()

    if LOSS_TYPE == "NLLLoss":
        return nn.NLLLoss()

    raise ValueError(f"Loss '{LOSS_TYPE}' is not supported.")





def create_optimizer(model):
    if OPTIMIZER_TYPE == "SGD":
        return torch.optim.SGD(
            model.parameters(),
            lr=LR,
            momentum=MOMENTUM,
            weight_decay=WD
        )
    
    if OPTIMIZER_TYPE == "Adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=LR,
            weight_decay=WD
        )
    
    if OPTIMIZER_TYPE == "AdamW":
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WD
        )
    
    raise ValueError(f"Optimizer '{OPTIMIZER_TYPE}' is not supported.")







def create_scheduler(optimizer):

    if SCHEDULER_TYPE == "StepLR":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=STEP_SIZE,
            gamma=GAMMA
        )

    if SCHEDULER_TYPE == "MultiStepLR":
        return torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=MILESTONES,
            gamma=GAMMA
        )

    if SCHEDULER_TYPE == "CosineAnnealingLR":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=T_MAX,
            eta_min=ETA_MIN
        )

    if SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=T_0,          
            T_mult=T_MULT,    
            eta_min=ETA_MIN
        )
    
    if SCHEDULER_TYPE == "ReduceLROnPlateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",         
            factor=FACTOR,
            patience=PATIENCE,
            threshold=THRESHOLD,
            min_lr=MIN_LR
        )

    raise ValueError(f"Scheduler '{SCHEDULER_TYPE}' is not supported.")




In [ ]:

def build_transform_from_config(transforms_cfg):

    transforms_list = []
    
    for tr_config in transforms_cfg:
        name = tr_config.get("name", "")
        
        if name == "RandomCrop":
            size = tr_config.get("size")
            padding = tr_config.get("padding", 0)
            transforms_list.append(T.RandomCrop(size, padding=padding))
            
        elif name == "RandomResizedCrop":
            size = tr_config.get("size")
            transforms_list.append(T.RandomResizedCrop(size))
            
        elif name == "RandomHorizontalFlip":
            p = tr_config.get("p", 0.5)
            transforms_list.append(T.RandomHorizontalFlip(p))
            
        elif name == "Resize":
            size = tr_config.get("size")
            transforms_list.append(T.Resize(size))
            
        elif name == "CenterCrop":
            size = tr_config.get("size")
            transforms_list.append(T.CenterCrop(size))
            
        elif name == "ToTensor":
            transforms_list.append(T.ToTensor())
            
        elif name == "Normalize":
            mean = tr_config.get("mean", [0.485, 0.456, 0.406])
            std = tr_config.get("std", [0.229, 0.224, 0.225])
            transforms_list.append(T.Normalize(mean=mean, std=std))
            
        else:
            raise ValueError(f"Unsupported transform: {name}")

    
    return T.Compose(transforms_list)

# ============================================================================
# BUILD TRANSFORMS FROM EXTRACTED STUDENT CE-ONLY CONFIGS
# ============================================================================
transform_train = build_transform_from_config(TRAIN_TRANSFORMS_CFG)
transform_test = build_transform_from_config(TEST_TRANSFORMS_CFG)

print(f"[OK] Transforms loaded from Student CE-only metrics\n")
print(f"Train transforms: {TRAIN_TRANSFORMS_CFG}\n")
print(f"Test transforms: {TEST_TRANSFORMS_CFG}\n")


In [ ]:
if DATASET_NAME=="CIFAR10":

    train_dataset_full=torchvision.datasets.CIFAR10(root=DATA,train=True,download=True,transform=transform_train)
    val_dataset_full=torchvision.datasets.CIFAR10(root=DATA,train=True,download=True,transform=transform_test)
    test_dataset=torchvision.datasets.CIFAR10(root=DATA,train=False,download=True,transform=transform_test)

elif DATASET_NAME=="CIFAR100":

    train_dataset_full=torchvision.datasets.CIFAR100(root=DATA,train=True,download=True,transform=transform_train)
    val_dataset_full=torchvision.datasets.CIFAR100(root=DATA,train=True,download=True,transform=transform_test)
    test_dataset=torchvision.datasets.CIFAR100(root=DATA,train=False,download=True,transform=transform_test)

elif DATASET_NAME=="Food101":

    train_dataset_full=torchvision.datasets.Food101(root=DATA,split="train",download=True,transform=transform_train)
    val_dataset_full=torchvision.datasets.Food101(root=DATA,split="train",download=True,transform=transform_test)
    test_dataset=torchvision.datasets.Food101(root=DATA,split="test",download=True,transform=transform_test)

elif DATASET_NAME=="Flowers102":

    train_dataset_full=torchvision.datasets.Flowers102(root=DATA,split="train",download=True,transform=transform_train)
    val_dataset_full=torchvision.datasets.Flowers102(root=DATA,split="val",download=True,transform=transform_test)
    test_dataset=torchvision.datasets.Flowers102(root=DATA,split="test",download=True,transform=transform_test)

else:
    raise ValueError(f"Unsupported dataset: {DATASET_NAME}")

In [ ]:
if hasattr(train_dataset_full, "class_to_idx"):
    class_to_idx=train_dataset_full.class_to_idx
else:
    class_to_idx={str(i): i for i in range(NUM_CLASSES)}
idx_to_class={v:k for k,v in class_to_idx.items()}

In [ ]:
def build_model(arch_name, num_classes):

    if PRETRAINED_MODEL:
        model_dict = {
            "ResNet34": (models.resnet34, ResNet34_Weights.DEFAULT),
            "ViT_B_16": (models.vit_b_16, ViT_B_16_Weights.DEFAULT)
        }

        # timm models (ViT-Small)
        timm_models = {
            "ViT_Small": "vit_small_patch16_224"
        }

        if arch_name in timm_models:
            model = timm.create_model(timm_models[arch_name], pretrained=True, num_classes=num_classes)
            return model.to(device)

        if arch_name not in model_dict:
            raise ValueError(f"Unsupported architecture: {arch_name}")

        model_fn, weights = model_dict[arch_name]
        model = model_fn(weights=weights)

    else:
        model_dict = {
            "ResNet18": models.resnet18,
            "ResNet50": models.resnet50
        }

        if arch_name not in model_dict:
            raise ValueError(f"Unsupported architecture: {arch_name}")

        model_fn = model_dict[arch_name]
        model = model_fn(weights=None)

    if hasattr(model, "heads"):
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
    else:
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    return model.to(device)

In [ ]:
teacher=build_model(TEACHER_NAME,NUM_CLASSES)

teacher_model_path=os.path.join(TEACHER_PATH,"model.pth")

teacher.load_state_dict(torch.load(teacher_model_path,map_location=device))
teacher.eval()

print(f"[OK] Teacher model {TEACHER_NAME} loaded from: {teacher_model_path}\n")

In [ ]:

def load_selection_indices(idx_to_class,method):

    if method=="Random":

        all_seeds_indices={}

        for seed in SEEDS:

            seed_path=os.path.join(RANDOM_SELECTION_PATH,f"seed_{seed}",f"split_{SPLIT}",f"k_{K}")
            class_indices={}

            for class_id,class_name in idx_to_class.items():

                file_path=os.path.join(seed_path,f"class_{class_id}_{class_name}_indices.npy")

                if os.path.exists(file_path):

                    indices=np.load(file_path)
                    class_indices[class_id]=indices
                else:
                    class_indices[class_id]=np.array([],dtype=int)

            all_seeds_indices[seed]=class_indices

        return all_seeds_indices

    elif method in MULTI_SEED_METHODS:

        all_seeds_indices={}

        for seed in SEEDS:

            seed_path=os.path.join(MULTI_SEED_SELECTION_PATH,f"seed_{seed}",f"split_{SPLIT}",f"k_{K}")
            class_indices={}

            for class_id,class_name in idx_to_class.items():

                file_path=os.path.join(seed_path,f"class_{class_id}_{class_name}_indices.npy")

                if os.path.exists(file_path):

                    indices=np.load(file_path)
                    class_indices[class_id]=indices
                else:
                    class_indices[class_id]=np.array([],dtype=int)

            all_seeds_indices[seed]=class_indices

        return all_seeds_indices
    
    else:
        class_indices={}

        for class_id,class_name in idx_to_class.items():

            file_path=os.path.join(DETERMINISTIC_SELECTION_PATH,f"seed_{SEED}",f"split_{SPLIT}",f"k_{K}",f"class_{class_id}_{class_name}_indices.npy")

            if os.path.exists(file_path):

                indices=np.load(file_path)
                class_indices[class_id]=indices
            else:
                class_indices[class_id]=np.array([],dtype=int)
                
        return class_indices


In [ ]:

def save_aggregate_metrics(save_dir, filename, best_run_seed, val_accs, val_top5s, val_losses, test_accs, test_losses, test_top5s, kd_gains):

    aggregate = {
        "k_per_class": K,
        "seeds": SEEDS,
        "best_run_seed": best_run_seed,
        "val_acc_mean": float(np.mean(val_accs)),
        "val_acc_std": float(np.std(val_accs, ddof=1)),
        "val_top5_mean": float(np.mean(val_top5s)),
        "val_top5_std": float(np.std(val_top5s, ddof=1)),
        "val_loss_mean": float(np.mean(val_losses)),
        "val_loss_std": float(np.std(val_losses, ddof=1)),
        "test_acc_mean": float(np.mean(test_accs)),
        "test_acc_std": float(np.std(test_accs, ddof=1)),
        "test_loss_mean": float(np.mean(test_losses)),
        "test_top5_mean": float(np.mean(test_top5s)),
        "test_top5_std": float(np.std(test_top5s, ddof=1)),
        "kd_gain_mean": float(np.mean(kd_gains)),
        "kd_gain_std": float(np.std(kd_gains, ddof=1)),
        "per_run": [
            {"seed": s, "val_acc": va, "val_top5": vt5, "val_loss": vl, "test_acc": ta, "test_loss": tl, "test_top5": t5}
            for s, va, vt5, vl, ta, tl, t5 in zip(SEEDS, val_accs, val_top5s, val_losses, test_accs, test_losses, test_top5s)
        ]
    }

    with open(os.path.join(save_dir, filename), "w") as f:
        json.dump(aggregate, f, indent=2)

    print(f"[OK] Saved {filename} in {save_dir}\n")







def transforms_to_config(compose_transform):
    cfg = []
    for tr in compose_transform.transforms:
        if isinstance(tr, T.RandomResizedCrop):
            cfg.append({"name": "RandomResizedCrop", "size": tr.size})
        elif isinstance(tr, T.RandomCrop):
            cfg.append({"name": "RandomCrop", "size": tr.size, "padding": tr.padding})
        elif isinstance(tr, T.RandomHorizontalFlip):
            cfg.append({"name": "RandomHorizontalFlip", "p": tr.p})
        elif isinstance(tr, T.Resize):
            cfg.append({"name": "Resize", "size": tr.size})
        elif isinstance(tr, T.CenterCrop):
            cfg.append({"name": "CenterCrop", "size": tr.size})
        elif isinstance(tr, T.ToTensor):
            cfg.append({"name": "ToTensor"})
        elif isinstance(tr, T.Normalize):
            cfg.append({"name": "Normalize", "mean": list(tr.mean), "std": list(tr.std)})
        else:
            raise ValueError(f"Unsupported transform for saving: {tr}")
        
    return cfg








def save_experiment(experiment_dir):

    # =========================
    # OPTIMIZER
    # =========================
    optimizer_params = {}
    if OPTIMIZER_TYPE == "SGD":
        optimizer_params = {
            "lr": LR,
            "momentum": MOMENTUM,
            "weight_decay": WD
        }
    elif OPTIMIZER_TYPE in ["Adam", "AdamW"]:
        optimizer_params = {
            "lr": LR,
            "weight_decay": WD
        }

    # =========================
    # SCHEDULER
    # =========================
    scheduler_params = {}
    if SCHEDULER_TYPE == "StepLR":
        scheduler_params = {"step_size": STEP_SIZE, "gamma": GAMMA}
    elif SCHEDULER_TYPE == "MultiStepLR":
        scheduler_params = {"milestones": MILESTONES, "gamma": GAMMA}
    elif SCHEDULER_TYPE == "CosineAnnealingLR":
        scheduler_params = {"t_max": T_MAX, "eta_min": ETA_MIN}
    elif SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        scheduler_params = {"t_0": T_0, "t_mult": T_MULT, "eta_min": ETA_MIN}
    elif SCHEDULER_TYPE == "ReduceLROnPlateau":
        scheduler_params = {
            "factor": FACTOR,
            "patience": PATIENCE,
            "threshold": THRESHOLD,
            "min_lr": MIN_LR,
            "mode": "min"
        }

    # =========================
    # LOSS FUNCTION
    # =========================
    loss_params = {}
    if LOSS_TYPE == "CrossEntropyLoss":
        loss_params = {
            "label_smoothing": LABEL_SMOOTHING
        }

    config_metrics = {
        "teacher_model_name": TEACHER_NAME,
        "student_model_name": STUDENT_NAME,
        "dataset_name": DATASET_NAME,
        "method": METHOD,
        "selection_k": K,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "split": SPLIT,
        "seed": SEED,
        "run_seeds": SEEDS,
        "distillation_type": DISTILLATION_TYPE,
        "temperature": TEMP,
        "soft_target_loss_weight": SOFT_TARGET_LOSS_WEIGHT,
        "criterion_loss_weight": CRITERION_LOSS_WEIGHT,
        "optimizer": {
            "type": OPTIMIZER_TYPE,
            "params": optimizer_params
        },
        "scheduler": {
            "type": SCHEDULER_TYPE,
            "params": scheduler_params
        },
        "loss_function": {
            "type": LOSS_TYPE,
            "params": loss_params
        },
        "transforms_train": transforms_to_config(transform_train),
        "transforms_test": transforms_to_config(transform_test)
    }

    with open(os.path.join(experiment_dir, "metrics.json"), "w") as f:
        json.dump(config_metrics, f, indent=2)

    print(f"[OK] Saved metrics.json in {experiment_dir}\n")
    print(f"[OK] All done for K={K} in {experiment_dir}\n")




    
def save_plot_training_curves(run_dir, train_losses, val_losses, train_accs, val_accs):
    
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, train_accs, label="Train Accuracy", linewidth=2)
    plt.plot(epochs, val_accs, label="Validation Accuracy", linewidth=2)
    plt.title(f"{STUDENT_NAME} - {DATASET_NAME}", fontsize=14)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy (%)", fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "plot_accuracy.pdf"), dpi=300)
    plt.close()
    
    plt.figure(figsize=(8, 6))
    plt.plot(epochs, train_losses, label="Train Loss", linewidth=2)
    plt.plot(epochs, val_losses, label="Validation Loss", linewidth=2)
    plt.title(f"{STUDENT_NAME} - {DATASET_NAME}", fontsize=14)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "plot_loss.pdf"), dpi=300)
    plt.close()
    
    print(f"[OK] Plots saved as: plot_accuracy.pdf and plot_loss.pdf in {run_dir}\n")










def save_confusion_matrix(run_dir, y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)
    cmn = cm / np.maximum(1, cm.sum(axis=1, keepdims=True))  
    perc = cmn * 100.0
    perc_r = np.round(perc, 1)                             

   
    for i in range(perc_r.shape[0]):
        diff = 100.0 - perc_r[i].sum()
        if abs(diff) >= 1e-9:
            j = int(np.argmax(perc[i]))                   
            perc_r[i, j] += diff                           

    annot = np.vectorize(lambda x: f"{x:.1f}%")(perc_r)

    plt.figure(figsize=(8, 6))
    sns.heatmap(perc_r, annot=annot, fmt="", cmap="Blues", cbar=False, vmin=0, vmax=100)
    plt.title(f"{STUDENT_NAME} - {DATASET_NAME}", fontsize=14)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "cm_test.pdf"), dpi=300)
    plt.close()

    print(f"[OK] Confusion matrix saved as: cm_test.pdf in {run_dir}\n")











def save_per_class_accuracy_sorted(run_dir, y_true, y_pred, k, mode):

    C = int(NUM_CLASSES)

    
    cm = confusion_matrix(y_true, y_pred, labels=range(C)).astype(float)
    support = cm.sum(axis=1)
    acc = np.where(support > 0, np.diag(cm) / support, 0.0) * 100.0

    idx_valid = np.where(support > 0)[0]

    if len(idx_valid) == 0:

        print("[WARN] No classes with support>0; nothing to plot")
        return

    k = int(min(k, len(idx_valid)))
    acc_valid = acc[idx_valid]

    if mode == "bottom":

        sel_in_valid = np.argsort(acc_valid)[:k]
        title = f"{STUDENT_NAME} - {DATASET_NAME}"
        fname = f"per_class_test_bottom{k}.pdf"

    else:

        sel_in_valid = np.argsort(-acc_valid)[:k]
        title = f"{STUDENT_NAME} - {DATASET_NAME}"
        fname = f"per_class_test_top{k}.pdf"

    order = idx_valid[sel_in_valid]     
    labels = [str(i) for i in order]
    x = np.arange(k)

    fig = plt.figure(figsize=(max(10, k * 0.6), 4))
    ax = plt.gca()
    ax.bar(x, acc[order])
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)   
    ax.set_yticks(np.arange(0, 101, 10))      
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(title)
    ax.set_ylim(0, 100)

    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, fname), dpi=300)
    plt.close(fig)

    print(f"[OK] Per-class {mode} accuracy plot saved as: {fname} in {run_dir}\n")






def save_confusion_matrix_sparse(run_dir, y_true, y_pred, step):

    C = len(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=range(C)).astype(float)
    cmn = (cm / np.maximum(1, cm.sum(axis=1, keepdims=True))) * 100.0

    plt.figure(figsize=(10, 10))
    im = plt.imshow(np.clip(cmn, 0, 100), interpolation="nearest", aspect="auto", cmap="Blues", vmin=0, vmax=100)
    cb = plt.colorbar(im)
    cb.set_label("%")

    ticks = np.arange(0, C, step)
    tick_labels = [str(i) for i in ticks]
    plt.xticks(ticks, tick_labels, fontsize=8)
    plt.yticks(ticks, tick_labels, fontsize=8)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{STUDENT_NAME} - {DATASET_NAME}")
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "cm_test.pdf"), dpi=300)
    plt.close()

    print(f"[OK] Confusion matrix saved as: cm_test.pdf in {run_dir}\n")






def extract_embeddings_fc_input(model, loader, max_samples):
    model.eval()
    feats = []
    labs = []

    def hook_fn(module, inputs, outputs):
        feats.append(inputs[0].detach().cpu().flatten(1))

    if hasattr(model, "heads"):
        classifier = model.heads.head
    elif hasattr(model, "head"):
        classifier = model.head
    else:
        classifier = model.fc
    handle = classifier.register_forward_hook(hook_fn)

    seen = 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting embeddings"):
            images = images.to(device)
            labels = labels.to(device)

            _ = model(images)

            labs.append(labels.detach().cpu())
            seen += labels.size(0)

            if seen >= max_samples:
                break

    handle.remove()

    X = torch.cat(feats, dim=0)[:max_samples].numpy()
    y = torch.cat(labs, dim=0)[:max_samples].numpy()

    return X, y








def save_tsne_plot(run_dir, Z, y):

    if NUM_CLASSES <= 10:
        cmap = "tab10"
    else:
        cmap = "gist_ncar"

    plt.figure(figsize=(8, 6))
    plt.scatter(Z[:, 0], Z[:, 1], c=y, s=8, cmap=cmap, alpha=0.8)
    plt.title(f"{STUDENT_NAME} - {DATASET_NAME}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "tsne_test.pdf"), dpi=300)
    plt.close()

    print(f"\n[OK] t-SNE plot saved as: tsne_test.pdf in {run_dir}\n")


In [ ]:
def train_one_epoch_distillation(teacher,student,train_loader,optimizer,criterion):
    
    student.train()
    teacher.eval()

    running_loss=0.0
    correct=0
    total=0

    for images,labels in tqdm(train_loader,desc=f"Distilling {TEACHER_NAME} → {STUDENT_NAME}"):

        images,labels=images.to(device),labels.to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            teacher_logits=teacher(images)

        student_logits=student(images)

        soft_targets = nn.functional.softmax(teacher_logits / TEMP, dim=-1)
        soft_prob = nn.functional.log_softmax(student_logits / TEMP, dim=-1)  
        soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (TEMP**2)

        label_loss=criterion(student_logits,labels)

        loss=SOFT_TARGET_LOSS_WEIGHT*soft_targets_loss+CRITERION_LOSS_WEIGHT*label_loss

        loss.backward()
        optimizer.step()

        running_loss+= loss.item()*images.size(0)
        predicted=torch.argmax(student_logits,dim=1)

        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()

    epoch_loss=running_loss/total
    epoch_acc=100.*correct/total

    return epoch_loss,epoch_acc


In [ ]:
def evaluate_distillation(model,validation_loader,criterion):
    
    model.eval()
    
    running_loss=0.0
    correct=0
    total=0
    running_top5=0.0

    with torch.no_grad():
        for images,labels in tqdm(validation_loader,desc="Validation"):

            images,labels=images.to(device),labels.to(device)
            outputs=model(images)

            loss=criterion(outputs,labels)

            running_loss+=loss.item()*images.size(0)     
            predicted=torch.argmax(outputs,dim=1)

            total+=labels.size(0)
            correct+=(predicted==labels).sum().item()

            top5=outputs.topk(5, dim=1).indices
            running_top5+=top5.eq(labels.view(-1,1)).any(dim=1).float().sum().item()
            
    epoch_loss=running_loss/total
    epoch_acc=100.*correct/total
    epoch_top5=100.*running_top5/total   

    return epoch_loss,epoch_acc,epoch_top5

In [ ]:
def train_knowledge_distillation(teacher, student, train_loader, validation_loader, criterion, optimizer, scheduler):
        
    best_model_val_acc = 0.0
    best_model_val_loss = float("inf")
    best_epoch = 0
    best_val_top5 = 0.0
    best_model = None

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    print(f"Starting Knowledge Distillation between teacher {TEACHER_NAME} and student {STUDENT_NAME} on {DATASET_NAME} using {METHOD} for {EPOCHS} epochs:\n")

    for epoch in range(EPOCHS):

        train_loss, train_acc = train_one_epoch_distillation(teacher, student, train_loader, optimizer, criterion)
        val_loss, val_acc, val_top5 = evaluate_distillation(student, validation_loader, criterion)

        if SCHEDULER_TYPE == "ReduceLROnPlateau":
            scheduler.step(val_loss)
        else:
            scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"{STUDENT_NAME} | Epoch [{epoch+1}/{EPOCHS}] | Train Acc: {train_acc:.2f}% | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Top5: {val_top5}\n")

        if (val_acc > best_model_val_acc) or (
            val_acc == best_model_val_acc and val_loss < best_model_val_loss
        ):
            best_model_val_acc = val_acc
            best_model_val_loss = val_loss
            best_val_top5 = val_top5
            best_epoch = epoch + 1
            best_model = copy.deepcopy(student)

    print(f"[OK] Knowledge Distillation completed for {STUDENT_NAME} on {DATASET_NAME} using {METHOD}")
    print(f"Best validation accuracy: {best_model_val_acc:.2f}% at epoch {best_epoch} | Best Val Top-5: {best_val_top5:.2f}%\n")

    return best_model, best_model_val_acc, best_model_val_loss, train_losses, val_losses, train_accs, val_accs, best_epoch, best_val_top5


In [ ]:
def test_distillation(model,test_loader,criterion):
    
    model.eval()
    
    running_loss=0.0
    correct=0
    total=0
    running_top5=0.0

    y_true=[]
    y_pred=[]

    print(f"Starting testing for {STUDENT_NAME} on {DATASET_NAME}:\n")

    with torch.no_grad():
        for images,labels in tqdm(test_loader,desc="Testing"):

            images,labels=images.to(device),labels.to(device)
            outputs=model(images)

            loss=criterion(outputs,labels)
            running_loss+=loss.item()*images.size(0)     

            predicted=torch.argmax(outputs,dim=1)
            total+=labels.size(0)
            correct+=(predicted==labels).sum().item()

            top5=outputs.topk(5,dim=1).indices
            running_top5+=top5.eq(labels.view(-1,1)).any(dim=1).float().sum().item()

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    test_loss=running_loss/total
    test_acc=100.*correct/total        
    test_top5=100.*running_top5/total   

    print(f"\n[OK] Test completed for {STUDENT_NAME} on {DATASET_NAME} | Test Acc: {test_acc:.2f}% | Test Loss: {test_loss:.4f} | Test Top-5: {test_top5:.2f}%\n")

    return test_loss,test_acc,test_top5,y_true,y_pred


In [ ]:

val_indices = np.load(os.path.join(SPLIT_DIR, "val_indices.npy"))
train_indices = np.load(os.path.join(SPLIT_DIR, "train_indices.npy"))

NUM_WORKERS = 8

# ==================== SHARED LOADERS (same for all runs) ====================
validation_dataset = Subset(val_dataset_full, val_indices)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# ==================== t-SNE INDICES (shared) ====================
tsne_indices = np.load(os.path.join(STUDENT_CE_ONLY_PATH, "tsne_test_indices.npy"))
tsne_dataset = Subset(test_dataset, tsne_indices)
tsne_loader = DataLoader(tsne_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"[OK] Loaded tsne_test_indices.npy from {STUDENT_CE_ONLY_PATH} ({len(tsne_indices)} samples)\n")

# ============================================================================
# LOOP OVER ALL K VALUES
# ============================================================================
for K in K_LIST:

    print(f"\n{'#'*70}")
    print(f"#  K = {K}  ({K_LIST.index(K)+1}/{len(K_LIST)})")
    print(f"{'#'*70}\n")

    # ==================== ROOT_DIR (depends on K) ====================
    ROOT_DIR = os.path.join(
        f"teacher_{TEACHER_NAME}",
        f"student_{STUDENT_NAME}",
        DATASET_NAME,
        f"seed_{SEED}",
        f"split_{SPLIT}",
        f"k_{K}",
        METHOD
    )

    if METHOD == "Random":
        # ==================== RANDOM ====================
        # 5 selections (one per seed), 5 runs with corresponding seeds

        random_indices = load_selection_indices(idx_to_class, "Random")

        rand_val_accs, rand_val_losses = [], []
        rand_val_top5s = []
        rand_test_accs, rand_test_losses = [], []
        rand_test_top5s = []
        rand_kd_gains = []

        rand_best_val_acc = -1.0
        rand_best_val_loss = float("inf")
        rand_best_val_top5 = 0.0
        rand_best_run_seed = None
        rand_best_run_model = None
        rand_best_run_train_losses = None
        rand_best_run_val_losses = None
        rand_best_run_train_accs = None
        rand_best_run_val_accs = None
        rand_best_run_y_true = None
        rand_best_run_y_pred = None

        for i, run_seed in enumerate(SEEDS):

            set_seed(run_seed)

            class_indices = random_indices[run_seed]
            rand_indices_selected = train_indices[np.concatenate(list(class_indices.values()))]

            print(f"\n{'='*60}")
            print(f"[Random] K={K} | Run {i+1}/{len(SEEDS)} | Seed: {run_seed}")
            print(f"{'='*60}\n")

            student = build_model(STUDENT_NAME, NUM_CLASSES)
            student.train()

            train_dataset = Subset(train_dataset_full, rand_indices_selected)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

            criterion = create_loss()
            optimizer = create_optimizer(student)
            scheduler = create_scheduler(optimizer)

            best_model, best_acc, best_loss, train_losses, val_losses, train_accs, val_accs, best_epoch, best_top5 = train_knowledge_distillation(
                teacher=teacher, student=student, train_loader=train_loader,
                validation_loader=validation_loader, criterion=criterion,
                optimizer=optimizer, scheduler=scheduler
            )

            test_loss, test_acc, test_top5, y_true_test, y_pred_test = test_distillation(
                model=best_model, test_loader=test_loader, criterion=criterion
            )

            rand_val_accs.append(best_acc)
            rand_val_losses.append(best_loss)
            rand_val_top5s.append(best_top5)
            rand_test_accs.append(test_acc)
            rand_test_losses.append(test_loss)
            rand_test_top5s.append(test_top5)
            rand_kd_gains.append(test_acc - TEST_STUDENT_CE_ACC)

            if (best_acc > rand_best_val_acc) or (best_acc == rand_best_val_acc and best_loss < rand_best_val_loss):
                rand_best_val_acc = best_acc
                rand_best_val_loss = best_loss
                rand_best_val_top5 = best_top5
                rand_best_run_seed = run_seed
                rand_best_run_model = copy.deepcopy(best_model)
                rand_best_run_train_losses = train_losses
                rand_best_run_val_losses = val_losses
                rand_best_run_train_accs = train_accs
                rand_best_run_val_accs = val_accs
                rand_best_run_y_true = y_true_test
                rand_best_run_y_pred = y_pred_test

        print(f"Best random run (K={K}): seed={rand_best_run_seed} | Val Acc={rand_best_val_acc:.2f}% | Val Loss={rand_best_val_loss:.4f} | Val Top-5={rand_best_val_top5:.2f}%\n")

        # ==================== EXPERIMENT FOLDER ====================
        if DISTILLATION_TYPE == "soft_label":
            EXPERIMENT_FOLDER = (
                f"val_acc={rand_best_val_acc:.2f}%"
                f"_val_loss={rand_best_val_loss:.4f}"
                f"_soft_weight={SOFT_TARGET_LOSS_WEIGHT}"
                f"_cr_weight={CRITERION_LOSS_WEIGHT}"
                f"_T={TEMP}"
                f"_lr={LR}"
                f"_wd={WD}"
                f"_ep={EPOCHS}"
                f"_bs={BATCH_SIZE}"
                f"_opt={OPTIMIZER_TYPE}"
                f"_sch={SCHEDULER_TYPE}"
            )
        elif DISTILLATION_TYPE == "feature_map":
            EXPERIMENT_FOLDER = (
                f"val_acc={rand_best_val_acc:.2f}%"
                f"_val_loss={rand_best_val_loss:.4f}"
                f"_featuremap_weight={FEATURE_MAP_WEIGHT}"
                f"_lr={LR}"
                f"_wd={WD}"
                f"_ep={EPOCHS}"
                f"_bs={BATCH_SIZE}"
                f"_opt={OPTIMIZER_TYPE}"
                f"_sch={SCHEDULER_TYPE}"
            )

        EXPERIMENT_DIR = os.path.join(ROOT_DIR, EXPERIMENT_FOLDER)
        os.makedirs(EXPERIMENT_DIR, exist_ok=True)
        print(f"[OK] Experiment directory: {EXPERIMENT_DIR}\n")

        # ==================== SAVE RANDOM ====================

        rand_model_path = os.path.join(EXPERIMENT_DIR, f"model_random_seed_{rand_best_run_seed}.pth")
        torch.save(rand_best_run_model.state_dict(), rand_model_path)
        print(f"[OK] Random model saved: {rand_model_path}\n")

        save_plot_training_curves(run_dir=EXPERIMENT_DIR, train_losses=rand_best_run_train_losses, val_losses=rand_best_run_val_losses, train_accs=rand_best_run_train_accs, val_accs=rand_best_run_val_accs)

        if DATASET_NAME == "CIFAR10":
            save_confusion_matrix(run_dir=EXPERIMENT_DIR, y_true=rand_best_run_y_true, y_pred=rand_best_run_y_pred)
        else:
            save_per_class_accuracy_sorted(run_dir=EXPERIMENT_DIR, y_true=rand_best_run_y_true, y_pred=rand_best_run_y_pred, k=20, mode="bottom")
            save_per_class_accuracy_sorted(run_dir=EXPERIMENT_DIR, y_true=rand_best_run_y_true, y_pred=rand_best_run_y_pred, k=20, mode="top")
            save_confusion_matrix_sparse(run_dir=EXPERIMENT_DIR, y_true=rand_best_run_y_true, y_pred=rand_best_run_y_pred, step=10)

        X, y = extract_embeddings_fc_input(model=rand_best_run_model, loader=tsne_loader, max_samples=len(tsne_indices))
        X_scaled = StandardScaler().fit_transform(X)
        tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED)
        Z = tsne.fit_transform(X_scaled)
        save_tsne_plot(run_dir=EXPERIMENT_DIR, Z=Z, y=y)

        save_aggregate_metrics(
            save_dir=EXPERIMENT_DIR,
            filename="random_metrics.json",
            best_run_seed=rand_best_run_seed,
            val_accs=rand_val_accs,
            val_top5s=rand_val_top5s,
            val_losses=rand_val_losses,
            test_accs=rand_test_accs,
            test_losses=rand_test_losses,
            test_top5s=rand_test_top5s,
            kd_gains=rand_kd_gains
        )

        save_experiment(EXPERIMENT_DIR)


    elif METHOD in MULTI_SEED_METHODS:
        # ==================== MULTI-SEED (KCENTER) ====================
        # 5 selections (one per seed), 5 runs with corresponding seeds

        multi_seed_indices = load_selection_indices(idx_to_class, METHOD)

        ms_val_accs, ms_val_losses = [], []
        ms_val_top5s = []
        ms_test_accs, ms_test_losses = [], []
        ms_test_top5s = []
        ms_kd_gains = []

        ms_best_val_acc = -1.0
        ms_best_val_loss = float("inf")
        ms_best_val_top5 = 0.0
        ms_best_run_seed = None
        ms_best_run_model = None
        ms_best_run_train_losses = None
        ms_best_run_val_losses = None
        ms_best_run_train_accs = None
        ms_best_run_val_accs = None
        ms_best_run_y_true = None
        ms_best_run_y_pred = None

        for i, run_seed in enumerate(SEEDS):

            set_seed(run_seed)

            class_indices = multi_seed_indices[run_seed]
            ms_indices_selected = train_indices[np.concatenate(list(class_indices.values()))]

            print(f"\n{'='*60}")
            print(f"[Multi-Seed] K={K} | Run {i+1}/{len(SEEDS)} | Seed: {run_seed}")
            print(f"{'='*60}\n")

            student = build_model(STUDENT_NAME, NUM_CLASSES)
            student.train()

            train_dataset = Subset(train_dataset_full, ms_indices_selected)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

            criterion = create_loss()
            optimizer = create_optimizer(student)
            scheduler = create_scheduler(optimizer)

            best_model, best_acc, best_loss, train_losses, val_losses, train_accs, val_accs, best_epoch, best_top5 = train_knowledge_distillation(
                teacher=teacher, student=student, train_loader=train_loader,
                validation_loader=validation_loader, criterion=criterion,
                optimizer=optimizer, scheduler=scheduler
            )

            test_loss, test_acc, test_top5, y_true_test, y_pred_test = test_distillation(
                model=best_model, test_loader=test_loader, criterion=criterion
            )

            ms_val_accs.append(best_acc)
            ms_val_losses.append(best_loss)
            ms_val_top5s.append(best_top5)
            ms_test_accs.append(test_acc)
            ms_test_losses.append(test_loss)
            ms_test_top5s.append(test_top5)
            ms_kd_gains.append(test_acc - TEST_STUDENT_CE_ACC)

            if (best_acc > ms_best_val_acc) or (best_acc == ms_best_val_acc and best_loss < ms_best_val_loss):
                ms_best_val_acc = best_acc
                ms_best_val_loss = best_loss
                ms_best_val_top5 = best_top5
                ms_best_run_seed = run_seed
                ms_best_run_model = copy.deepcopy(best_model)
                ms_best_run_train_losses = train_losses
                ms_best_run_val_losses = val_losses
                ms_best_run_train_accs = train_accs
                ms_best_run_val_accs = val_accs
                ms_best_run_y_true = y_true_test
                ms_best_run_y_pred = y_pred_test

        print(f"Best multi-seed run (K={K}): seed={ms_best_run_seed} | Val Acc={ms_best_val_acc:.2f}% | Val Loss={ms_best_val_loss:.4f} | Val Top-5={ms_best_val_top5:.2f}%\n")

        # ==================== EXPERIMENT FOLDER ====================
        if DISTILLATION_TYPE == "soft_label":
            EXPERIMENT_FOLDER = (
                f"val_acc={ms_best_val_acc:.2f}%"
                f"_val_loss={ms_best_val_loss:.4f}"
                f"_soft_weight={SOFT_TARGET_LOSS_WEIGHT}"
                f"_cr_weight={CRITERION_LOSS_WEIGHT}"
                f"_T={TEMP}"
                f"_lr={LR}"
                f"_wd={WD}"
                f"_ep={EPOCHS}"
                f"_bs={BATCH_SIZE}"
                f"_opt={OPTIMIZER_TYPE}"
                f"_sch={SCHEDULER_TYPE}"
            )
        elif DISTILLATION_TYPE == "feature_map":
            EXPERIMENT_FOLDER = (
                f"val_acc={ms_best_val_acc:.2f}%"
                f"_val_loss={ms_best_val_loss:.4f}"
                f"_featuremap_weight={FEATURE_MAP_WEIGHT}"
                f"_lr={LR}"
                f"_wd={WD}"
                f"_ep={EPOCHS}"
                f"_bs={BATCH_SIZE}"
                f"_opt={OPTIMIZER_TYPE}"
                f"_sch={SCHEDULER_TYPE}"
            )

        EXPERIMENT_DIR = os.path.join(ROOT_DIR, EXPERIMENT_FOLDER)
        os.makedirs(EXPERIMENT_DIR, exist_ok=True)
        print(f"[OK] Experiment directory: {EXPERIMENT_DIR}\n")

        # ==================== SAVE MULTI-SEED ====================

        ms_model_path = os.path.join(EXPERIMENT_DIR, f"model_multi_seed_seed_{ms_best_run_seed}.pth")
        torch.save(ms_best_run_model.state_dict(), ms_model_path)
        print(f"[OK] Multi-seed model saved: {ms_model_path}\n")

        save_plot_training_curves(run_dir=EXPERIMENT_DIR, train_losses=ms_best_run_train_losses, val_losses=ms_best_run_val_losses, train_accs=ms_best_run_train_accs, val_accs=ms_best_run_val_accs)

        if DATASET_NAME == "CIFAR10":
            save_confusion_matrix(run_dir=EXPERIMENT_DIR, y_true=ms_best_run_y_true, y_pred=ms_best_run_y_pred)
        else:
            save_per_class_accuracy_sorted(run_dir=EXPERIMENT_DIR, y_true=ms_best_run_y_true, y_pred=ms_best_run_y_pred, k=20, mode="bottom")
            save_per_class_accuracy_sorted(run_dir=EXPERIMENT_DIR, y_true=ms_best_run_y_true, y_pred=ms_best_run_y_pred, k=20, mode="top")
            save_confusion_matrix_sparse(run_dir=EXPERIMENT_DIR, y_true=ms_best_run_y_true, y_pred=ms_best_run_y_pred, step=10)

        X, y = extract_embeddings_fc_input(model=ms_best_run_model, loader=tsne_loader, max_samples=len(tsne_indices))
        X_scaled = StandardScaler().fit_transform(X)
        tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED)
        Z = tsne.fit_transform(X_scaled)
        save_tsne_plot(run_dir=EXPERIMENT_DIR, Z=Z, y=y)

        save_aggregate_metrics(
            save_dir=EXPERIMENT_DIR,
            filename="multi_seed_metrics.json",
            best_run_seed=ms_best_run_seed,
            val_accs=ms_val_accs,
            val_top5s=ms_val_top5s,
            val_losses=ms_val_losses,
            test_accs=ms_test_accs,
            test_losses=ms_test_losses,
            test_top5s=ms_test_top5s,
            kd_gains=ms_kd_gains
        )

        save_experiment(EXPERIMENT_DIR)


    else:
        # ==================== DETERMINISTIC (PCA) ====================
        # 1 selection (same indices for all runs), 5 runs with different seeds

        deterministic_indices = load_selection_indices(idx_to_class, METHOD)

        det_val_accs, det_val_losses = [], []
        det_val_top5s = []
        det_test_accs, det_test_losses = [], []
        det_test_top5s = []
        det_kd_gains = []

        det_indices_selected = train_indices[np.concatenate(list(deterministic_indices.values()))]

        det_best_val_acc = -1.0
        det_best_val_loss = float("inf")
        det_best_val_top5 = 0.0
        det_best_run_seed = None
        det_best_run_model = None
        det_best_run_train_losses = None
        det_best_run_val_losses = None
        det_best_run_train_accs = None
        det_best_run_val_accs = None
        det_best_run_y_true = None
        det_best_run_y_pred = None

        for i, run_seed in enumerate(SEEDS):

            set_seed(run_seed)

            print(f"\n{'='*60}")
            print(f"[Deterministic] K={K} | Run {i+1}/{len(SEEDS)} | Seed: {run_seed}")
            print(f"{'='*60}\n")

            student = build_model(STUDENT_NAME, NUM_CLASSES)
            student.train()

            train_dataset = Subset(train_dataset_full, det_indices_selected)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

            criterion = create_loss()
            optimizer = create_optimizer(student)
            scheduler = create_scheduler(optimizer)

            best_model, best_acc, best_loss, train_losses, val_losses, train_accs, val_accs, best_epoch, best_top5 = train_knowledge_distillation(
                teacher=teacher, student=student, train_loader=train_loader,
                validation_loader=validation_loader, criterion=criterion,
                optimizer=optimizer, scheduler=scheduler
            )

            test_loss, test_acc, test_top5, y_true_test, y_pred_test = test_distillation(
                model=best_model, test_loader=test_loader, criterion=criterion
            )

            det_val_accs.append(best_acc)
            det_val_losses.append(best_loss)
            det_val_top5s.append(best_top5)
            det_test_accs.append(test_acc)
            det_test_losses.append(test_loss)
            det_test_top5s.append(test_top5)
            det_kd_gains.append(test_acc - TEST_STUDENT_CE_ACC)

            if (best_acc > det_best_val_acc) or (best_acc == det_best_val_acc and best_loss < det_best_val_loss):
                det_best_val_acc = best_acc
                det_best_val_loss = best_loss
                det_best_val_top5 = best_top5
                det_best_run_seed = run_seed
                det_best_run_model = copy.deepcopy(best_model)
                det_best_run_train_losses = train_losses
                det_best_run_val_losses = val_losses
                det_best_run_train_accs = train_accs
                det_best_run_val_accs = val_accs
                det_best_run_y_true = y_true_test
                det_best_run_y_pred = y_pred_test

        print(f"Best deterministic run (K={K}): seed={det_best_run_seed} | Val Acc={det_best_val_acc:.2f}% | Val Loss={det_best_val_loss:.4f} | Val Top-5={det_best_val_top5:.2f}%\n")

        # ==================== EXPERIMENT FOLDER ====================
        if DISTILLATION_TYPE == "soft_label":
            EXPERIMENT_FOLDER = (
                f"val_acc={det_best_val_acc:.2f}%"
                f"_val_loss={det_best_val_loss:.4f}"
                f"_soft_weight={SOFT_TARGET_LOSS_WEIGHT}"
                f"_cr_weight={CRITERION_LOSS_WEIGHT}"
                f"_T={TEMP}"
                f"_lr={LR}"
                f"_wd={WD}"
                f"_ep={EPOCHS}"
                f"_bs={BATCH_SIZE}"
                f"_opt={OPTIMIZER_TYPE}"
                f"_sch={SCHEDULER_TYPE}"
            )
        elif DISTILLATION_TYPE == "feature_map":
            EXPERIMENT_FOLDER = (
                f"val_acc={det_best_val_acc:.2f}%"
                f"_val_loss={det_best_val_loss:.4f}"
                f"_featuremap_weight={FEATURE_MAP_WEIGHT}"
                f"_lr={LR}"
                f"_wd={WD}"
                f"_ep={EPOCHS}"
                f"_bs={BATCH_SIZE}"
                f"_opt={OPTIMIZER_TYPE}"
                f"_sch={SCHEDULER_TYPE}"
            )

        EXPERIMENT_DIR = os.path.join(ROOT_DIR, EXPERIMENT_FOLDER)
        os.makedirs(EXPERIMENT_DIR, exist_ok=True)
        print(f"[OK] Experiment directory: {EXPERIMENT_DIR}\n")

        # ==================== SAVE DETERMINISTIC ====================

        det_model_path = os.path.join(EXPERIMENT_DIR, f"model_deterministic_seed_{det_best_run_seed}.pth")
        torch.save(det_best_run_model.state_dict(), det_model_path)
        print(f"[OK] Deterministic model saved: {det_model_path}\n")

        save_plot_training_curves(run_dir=EXPERIMENT_DIR, train_losses=det_best_run_train_losses, val_losses=det_best_run_val_losses, train_accs=det_best_run_train_accs, val_accs=det_best_run_val_accs)

        if DATASET_NAME == "CIFAR10":
            save_confusion_matrix(run_dir=EXPERIMENT_DIR, y_true=det_best_run_y_true, y_pred=det_best_run_y_pred)
        else:
            save_per_class_accuracy_sorted(run_dir=EXPERIMENT_DIR, y_true=det_best_run_y_true, y_pred=det_best_run_y_pred, k=20, mode="bottom")
            save_per_class_accuracy_sorted(run_dir=EXPERIMENT_DIR, y_true=det_best_run_y_true, y_pred=det_best_run_y_pred, k=20, mode="top")
            save_confusion_matrix_sparse(run_dir=EXPERIMENT_DIR, y_true=det_best_run_y_true, y_pred=det_best_run_y_pred, step=10)

        X, y = extract_embeddings_fc_input(model=det_best_run_model, loader=tsne_loader, max_samples=len(tsne_indices))
        X_scaled = StandardScaler().fit_transform(X)
        tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED)
        Z = tsne.fit_transform(X_scaled)
        save_tsne_plot(run_dir=EXPERIMENT_DIR, Z=Z, y=y)

        save_aggregate_metrics(
            save_dir=EXPERIMENT_DIR,
            filename="deterministic_metrics.json",
            best_run_seed=det_best_run_seed,
            val_accs=det_val_accs,
            val_top5s=det_val_top5s,
            val_losses=det_val_losses,
            test_accs=det_test_accs,
            test_losses=det_test_losses,
            test_top5s=det_test_top5s,
            kd_gains=det_kd_gains
        )

        save_experiment(EXPERIMENT_DIR)

    print(f"\n[OK] Finished K={K}\n")

print(f"\n{'#'*70}")
print(f"#  ALL K VALUES COMPLETED: {K_LIST}")
print(f"{'#'*70}")
